In [ ]:
!pip install osmnx networkx folium ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.

In [ ]:
# -*- coding: utf-8 -*-
"""Time‑Aware Capacitated Vehicle Routing Problem – Structured (Fixed)

Now uses Euclidean‑based time fallback when no road path exists, ensuring feasibility.
"""

# ============================================================================
# 1.  INSTALLATION (uncomment and run once in Colab)
# ============================================================================
# !pip install osmnx networkx folium ortools

# ============================================================================
# 2.  IMPORTS
# ============================================================================
import osmnx as ox
import networkx as nx
import folium
import numpy as np
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

print("All libraries imported successfully!")

# ============================================================================
# 3.  CONFIGURATION
# ============================================================================
CENTER_POINT = (28.6129, 77.2295)   # India Gate, Delhi
RADIUS_METERS = 4000
NETWORK_TYPE = 'drive'

NUM_DELIVERIES = 30
RANDOM_SEED = 42

MAX_DEMAND = 9
SERVICE_TIME_PER_UNIT = 5 * 60      # seconds

VEHICLE_CAPACITIES = [85, 60, 65]
MAX_ROUTE_TIME = 14 * 60 * 60       # 14 hours

SOLVER_TIME_LIMIT_SECONDS = 10
FALLBACK_SPEED_KPH = 30             # km/h for Euclidean fallback

# ============================================================================
# 4.  HELPER FUNCTIONS
# ============================================================================

def load_road_network(center, radius, network_type):
    print(f"Downloading road network (radius {radius/1000:.1f} km) ...")
    G = ox.graph_from_point(center, dist=radius, network_type=network_type)
    print(f"  Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
    return G


def select_locations(G, num_deliveries, seed=None):
    if seed is not None:
        np.random.seed(seed)
    all_nodes = list(G.nodes)
    depot = np.random.choice(all_nodes, 1)[0]
    candidates = [n for n in all_nodes if n != depot]
    deliveries = np.random.choice(candidates, num_deliveries, replace=False)
    locations = [depot] + list(deliveries)
    node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in locations}
    print(f"Selected 1 depot and {num_deliveries} delivery points.")
    return locations, node_coords


def compute_distance_matrix(G, locations):
    n = len(locations)
    dist_matrix = np.zeros((n, n), dtype=np.int64)
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                dist_matrix[i][j] = 0
                continue
            try:
                path_length = nx.shortest_path_length(
                    G, locations[i], locations[j], weight='length'
                )
                dist_matrix[i][j] = int(path_length)
            except nx.NetworkXNoPath:
                fallback = euclidean_distance(i, j)
                dist_matrix[i][j] = fallback
                print(f"  ⚠️ No road path between {i} and {j}, using Euclidean: {fallback}m")
    return dist_matrix


def compute_travel_time_matrix(G, locations, fallback_speed_kph=30):
    """
    Compute travel times in seconds. If no road path exists, use Euclidean
    distance / (fallback speed) as a fallback.
    """
    # Add travel_time attribute to edges
    G_time = G.copy()
    for u, v, k, data in G_time.edges(keys=True, data=True):
        distance_m = data.get('length', 0)
        speed_kph = data.get('speed_kph', fallback_speed_kph)
        if isinstance(speed_kph, list):
            speed_kph = speed_kph[0]
        try:
            speed_kph = float(speed_kph)
        except (TypeError, ValueError):
            speed_kph = fallback_speed_kph
        speed_mps = speed_kph * 1000 / 3600
        travel_time = distance_m / speed_mps if speed_mps > 0 else 0
        data['travel_time'] = int(travel_time)

    n = len(locations)
    time_matrix = np.zeros((n, n), dtype=np.int64)

    # Precompute coords for Euclidean fallback
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                time_matrix[i][j] = 0
                continue
            try:
                tt = nx.shortest_path_length(
                    G_time, locations[i], locations[j], weight='travel_time'
                )
                time_matrix[i][j] = int(tt)
            except nx.NetworkXNoPath:
                # Fallback: Euclidean distance / fallback speed (m/s)
                distance_m = euclidean_distance(i, j)
                speed_mps = fallback_speed_kph * 1000 / 3600
                fallback_time = int(distance_m / speed_mps)
                time_matrix[i][j] = fallback_time
                print(f"  ⚠️ No travel‑time path between {i} and {j}, using Euclidean time: {fallback_time}s")
    return time_matrix


def generate_demands(num_locations, seed=None, max_demand=9):
    if seed is not None:
        np.random.seed(seed)
    demands = [0] + list(np.random.randint(1, max_demand+1, size=num_locations-1))
    # Convert to plain Python ints for cleaner printing
    demands = [int(d) for d in demands]
    print(f"Generated demands (total = {sum(demands)}): {demands}")
    return demands


def generate_service_times(demands, time_per_unit):
    service = [0] + [d * time_per_unit for d in demands[1:]]
    service = [int(s) for s in service]
    print(f"Service times (seconds): {service}")
    return service


def solve_time_aware_cvrp(time_matrix, demands, service_times,
                          vehicle_capacities, max_route_time,
                          time_limit=10, depot=0):
    num_vehicles = len(vehicle_capacities)
    num_nodes = len(time_matrix)

    data = {
        'time_matrix': time_matrix,
        'demands': demands,
        'service_times': service_times,
        'vehicle_capacities': vehicle_capacities,
        'num_vehicles': num_vehicles,
        'depot': depot,
        'max_route_time': max_route_time
    }

    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        travel = data['time_matrix'][from_node][to_node]
        service = data['service_times'][to_node]
        return int(travel + service)

    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity'
    )

    routing.AddDimension(
        transit_callback_index,
        0,
        data['max_route_time'],
        True,
        'Time'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = time_limit

    solution = routing.SolveWithParameters(search_parameters)
    if not solution:
        print("❌ No feasible solution found!")
        return None

    routes = {}
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(manager.IndexToNode(index))
        routes[vehicle_id] = route_nodes

    return {
        'routes': routes,
        'solution': solution,
        'routing': routing,
        'manager': manager,
        'time_dimension': routing.GetDimensionOrDie('Time')
    }


def visualize_routes(G, routes, locations, node_coords):
    depot_osm_id = locations[0]
    depot_lat, depot_lon = node_coords[depot_osm_id]

    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14,
                   tiles='CartoDB positron')

    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

    for idx, osm_id in enumerate(locations):
        lat, lon = node_coords[osm_id]
        if idx == 0:
            folium.Marker([lat, lon], popup="🏭 Depot",
                          icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:
            folium.Marker([lat, lon], popup=f"📍 Delivery {idx}",
                          icon=folium.Icon(color='gray', icon='info-sign')).add_to(m)

    for vehicle_id, route_indices in routes.items():
        color = colors[vehicle_id % len(colors)]
        route_coords = []
        for idx in route_indices:
            osm_id = locations[idx]
            lat, lon = node_coords[osm_id]
            route_coords.append((lat, lon))
        folium.PolyLine(route_coords, color=color, weight=4, opacity=0.7,
                        popup=f"🚚 Truck {vehicle_id+1}").add_to(m)

    return m


# ============================================================================
# 5.  MAIN
# ============================================================================

def main():
    G = load_road_network(CENTER_POINT, RADIUS_METERS, NETWORK_TYPE)

    locations, node_coords = select_locations(G, NUM_DELIVERIES, RANDOM_SEED)

    print("Computing distance matrix...")
    distance_matrix = compute_distance_matrix(G, locations)
    print(f"Distance matrix shape: {distance_matrix.shape}")

    print("Computing travel time matrix (with Euclidean fallback)...")
    time_matrix = compute_travel_time_matrix(G, locations, FALLBACK_SPEED_KPH)
    print(f"Time matrix shape: {time_matrix.shape}")
    print("First 5x5 entries (seconds):")
    print(time_matrix[:5, :5])

    demands = generate_demands(len(locations), RANDOM_SEED, MAX_DEMAND)
    service_times = generate_service_times(demands, SERVICE_TIME_PER_UNIT)

    print(f"\nSolving CVRP with {len(VEHICLE_CAPACITIES)} trucks, "
          f"capacities {VEHICLE_CAPACITIES}, max route time {MAX_ROUTE_TIME/3600:.1f} h ...")
    result = solve_time_aware_cvrp(
        time_matrix=time_matrix,
        demands=demands,
        service_times=service_times,
        vehicle_capacities=VEHICLE_CAPACITIES,
        max_route_time=MAX_ROUTE_TIME,
        time_limit=SOLVER_TIME_LIMIT_SECONDS
    )

    if result is None:
        print("Exiting due to no solution.")
        return

    routes = result['routes']
    solution = result['solution']
    routing = result['routing']
    manager = result['manager']
    time_dimension = result['time_dimension']

    total_distance = 0
    total_time = 0
    print("\n" + "=" * 60)
    for v_id, route in routes.items():
        load = sum(demands[node] for node in route)
        dist = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        total_distance += dist
        end_index = routing.End(v_id)
        route_time = solution.Value(time_dimension.CumulVar(end_index))
        total_time += route_time

        print(f"Truck {v_id+1}: {route}")
        print(f"  Load: {load} / {VEHICLE_CAPACITIES[v_id]}, "
              f"Distance: {dist/1000:.2f} km, "
              f"Time: {route_time/3600:.2f} h")
    print("=" * 60)
    print(f"📏 TOTAL COMBINED DISTANCE: {total_distance/1000:.2f} km")
    print(f"⏱️  TOTAL COMBINED VEHICLE TIME: {total_time/3600:.2f} hours")
    print("=" * 60)

    print("\nGenerating interactive map...")
    route_map = visualize_routes(G, routes, locations, node_coords)
    return route_map


# ============================================================================
# 6.  EXECUTE
# ============================================================================

map_object = main()
map_object

All libraries imported successfully!
  Nodes: 5229, Edges: 12292
Selected 1 depot and 30 delivery points.
Computing distance matrix...
  ⚠️ No road path between 0 and 23, using Euclidean: 3290m
  ⚠️ No road path between 1 and 23, using Euclidean: 4477m
  ⚠️ No road path between 2 and 23, using Euclidean: 4633m
  ⚠️ No road path between 3 and 23, using Euclidean: 3907m
  ⚠️ No road path between 4 and 23, using Euclidean: 3186m
  ⚠️ No road path between 5 and 23, using Euclidean: 5566m
  ⚠️ No road path between 6 and 23, using Euclidean: 4134m
  ⚠️ No road path between 7 and 23, using Euclidean: 6547m
  ⚠️ No road path between 8 and 23, using Euclidean: 7402m
  ⚠️ No road path between 9 and 23, using Euclidean: 7468m
  ⚠️ No road path between 10 and 23, using Euclidean: 7736m
  ⚠️ No road path between 11 and 23, using Euclidean: 4982m
  ⚠️ No road path between 12 and 23, using Euclidean: 6838m
  ⚠️ No road path between 13 and 23, using Euclidean: 4937m
  ⚠️ No road path between 14 and 23

In [ ]:
# -*- coding: utf-8 -*-
"""Time‑Aware Capacitated Vehicle Routing Problem – Structured (Fixed)

Now uses Euclidean‑based time fallback when no road path exists, ensuring feasibility.
"""

# ============================================================================
# 1.  INSTALLATION (uncomment and run once in Colab)
# ============================================================================
# !pip install osmnx networkx folium ortools

# ============================================================================
# 2.  IMPORTS
# ============================================================================
import osmnx as ox
import networkx as nx
import folium
import numpy as np
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

print("All libraries imported successfully!")

# ============================================================================
# 3.  CONFIGURATION
# ============================================================================
CENTER_POINT = (28.6129, 77.2295)   # India Gate, Delhi
RADIUS_METERS = 4000
NETWORK_TYPE = 'drive'

NUM_DELIVERIES = 30
RANDOM_SEED = 42

MAX_DEMAND = 9
SERVICE_TIME_PER_UNIT = 5 * 60      # seconds

VEHICLE_CAPACITIES = [85, 60, 65]
MAX_ROUTE_TIME = 14 * 60 * 60       # 14 hours

SOLVER_TIME_LIMIT_SECONDS = 10
FALLBACK_SPEED_KPH = 30             # km/h for Euclidean fallback

# ============================================================================
# 4.  HELPER FUNCTIONS
# ============================================================================

def load_road_network(center, radius, network_type):
    print(f"Downloading road network (radius {radius/1000:.1f} km) ...")
    G = ox.graph_from_point(center, dist=radius, network_type=network_type)
    print(f"  Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
    return G


def select_locations(G, num_deliveries, seed=None):
    if seed is not None:
        np.random.seed(seed)
    all_nodes = list(G.nodes)
    depot = np.random.choice(all_nodes, 1)[0]
    candidates = [n for n in all_nodes if n != depot]
    deliveries = np.random.choice(candidates, num_deliveries, replace=False)
    locations = [depot] + list(deliveries)
    node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in locations}
    print(f"Selected 1 depot and {num_deliveries} delivery points.")
    return locations, node_coords


def compute_distance_matrix(G, locations):
    n = len(locations)
    dist_matrix = np.zeros((n, n), dtype=np.int64)
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                dist_matrix[i][j] = 0
                continue
            try:
                path_length = nx.shortest_path_length(
                    G, locations[i], locations[j], weight='length'
                )
                dist_matrix[i][j] = int(path_length)
            except nx.NetworkXNoPath:
                fallback = euclidean_distance(i, j)
                dist_matrix[i][j] = fallback
                print(f"  ⚠️ No road path between {i} and {j}, using Euclidean: {fallback}m")
    return dist_matrix


def compute_travel_time_matrix(G, locations, fallback_speed_kph=30):
    """
    Compute travel times in seconds. If no road path exists, use Euclidean
    distance / (fallback speed) as a fallback.
    """
    # Add travel_time attribute to edges
    G_time = G.copy()
    for u, v, k, data in G_time.edges(keys=True, data=True):
        distance_m = data.get('length', 0)
        speed_kph = data.get('speed_kph', fallback_speed_kph)
        if isinstance(speed_kph, list):
            speed_kph = speed_kph[0]
        try:
            speed_kph = float(speed_kph)
        except (TypeError, ValueError):
            speed_kph = fallback_speed_kph
        speed_mps = speed_kph * 1000 / 3600
        travel_time = distance_m / speed_mps if speed_mps > 0 else 0
        data['travel_time'] = int(travel_time)

    n = len(locations)
    time_matrix = np.zeros((n, n), dtype=np.int64)

    # Precompute coords for Euclidean fallback
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                time_matrix[i][j] = 0
                continue
            try:
                tt = nx.shortest_path_length(
                    G_time, locations[i], locations[j], weight='travel_time'
                )
                time_matrix[i][j] = int(tt)
            except nx.NetworkXNoPath:
                # Fallback: Euclidean distance / fallback speed (m/s)
                distance_m = euclidean_distance(i, j)
                speed_mps = fallback_speed_kph * 1000 / 3600
                fallback_time = int(distance_m / speed_mps)
                time_matrix[i][j] = fallback_time
                print(f"  ⚠️ No travel‑time path between {i} and {j}, using Euclidean time: {fallback_time}s")
    return time_matrix


def generate_demands(num_locations, seed=None, max_demand=9):
    if seed is not None:
        np.random.seed(seed)
    demands = [0] + list(np.random.randint(1, max_demand+1, size=num_locations-1))
    # Convert to plain Python ints for cleaner printing
    demands = [int(d) for d in demands]
    print(f"Generated demands (total = {sum(demands)}): {demands}")
    return demands


def generate_service_times(demands, time_per_unit):
    service = [0] + [d * time_per_unit for d in demands[1:]]
    service = [int(s) for s in service]
    print(f"Service times (seconds): {service}")
    return service


def solve_time_aware_cvrp(time_matrix, demands, service_times,
                          vehicle_capacities, max_route_time,
                          time_limit=10, depot=0):
    num_vehicles = len(vehicle_capacities)
    num_nodes = len(time_matrix)

    data = {
        'time_matrix': time_matrix,
        'demands': demands,
        'service_times': service_times,
        'vehicle_capacities': vehicle_capacities,
        'num_vehicles': num_vehicles,
        'depot': depot,
        'max_route_time': max_route_time
    }

    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        travel = data['time_matrix'][from_node][to_node]
        service = data['service_times'][to_node]
        return int(travel + service)

    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity'
    )

    routing.AddDimension(
        transit_callback_index,
        0,
        data['max_route_time'],
        True,
        'Time'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = time_limit

    solution = routing.SolveWithParameters(search_parameters)
    if not solution:
        print("❌ No feasible solution found!")
        return None

    routes = {}
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(manager.IndexToNode(index))
        routes[vehicle_id] = route_nodes

    return {
        'routes': routes,
        'solution': solution,
        'routing': routing,
        'manager': manager,
        'time_dimension': routing.GetDimensionOrDie('Time')
    }


def visualize_routes(G, routes, locations, node_coords):
    depot_osm_id = locations[0]
    depot_lat, depot_lon = node_coords[depot_osm_id]

    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14,
                   tiles='CartoDB positron')

    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

    for idx, osm_id in enumerate(locations):
        lat, lon = node_coords[osm_id]
        if idx == 0:
            folium.Marker([lat, lon], popup="🏭 Depot",
                          icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:
            folium.Marker([lat, lon], popup=f"📍 Delivery {idx}",
                          icon=folium.Icon(color='gray', icon='info-sign')).add_to(m)

    for vehicle_id, route_indices in routes.items():
        color = colors[vehicle_id % len(colors)]
        route_coords = []
        for idx in route_indices:
            osm_id = locations[idx]
            lat, lon = node_coords[osm_id]
            route_coords.append((lat, lon))
        folium.PolyLine(route_coords, color=color, weight=4, opacity=0.7,
                        popup=f"🚚 Truck {vehicle_id+1}").add_to(m)

    return m


# ============================================================================
# 5.  MAIN
# ============================================================================

def main():
    G = load_road_network(CENTER_POINT, RADIUS_METERS, NETWORK_TYPE)

    locations, node_coords = select_locations(G, NUM_DELIVERIES, RANDOM_SEED)

    print("Computing distance matrix...")
    distance_matrix = compute_distance_matrix(G, locations)
    print(f"Distance matrix shape: {distance_matrix.shape}")

    print("Computing travel time matrix (with Euclidean fallback)...")
    time_matrix = compute_travel_time_matrix(G, locations, FALLBACK_SPEED_KPH)
    print(f"Time matrix shape: {time_matrix.shape}")
    print("First 5x5 entries (seconds):")
    print(time_matrix[:5, :5])

    demands = generate_demands(len(locations), RANDOM_SEED, MAX_DEMAND)
    service_times = generate_service_times(demands, SERVICE_TIME_PER_UNIT)

    print(f"\nSolving CVRP with {len(VEHICLE_CAPACITIES)} trucks, "
          f"capacities {VEHICLE_CAPACITIES}, max route time {MAX_ROUTE_TIME/3600:.1f} h ...")
    result = solve_time_aware_cvrp(
        time_matrix=time_matrix,
        demands=demands,
        service_times=service_times,
        vehicle_capacities=VEHICLE_CAPACITIES,
        max_route_time=MAX_ROUTE_TIME,
        time_limit=SOLVER_TIME_LIMIT_SECONDS
    )

    if result is None:
        print("Exiting due to no solution.")
        return

    routes = result['routes']
    solution = result['solution']
    routing = result['routing']
    manager = result['manager']
    time_dimension = result['time_dimension']

    total_distance = 0
    total_time = 0
    print("\n" + "=" * 60)
    for v_id, route in routes.items():
        load = sum(demands[node] for node in route)
        dist = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        total_distance += dist
        end_index = routing.End(v_id)
        route_time = solution.Value(time_dimension.CumulVar(end_index))
        total_time += route_time

        print(f"Truck {v_id+1}: {route}")
        print(f"  Load: {load} / {VEHICLE_CAPACITIES[v_id]}, "
              f"Distance: {dist/1000:.2f} km, "
              f"Time: {route_time/3600:.2f} h")
    print("=" * 60)
    print(f"📏 TOTAL COMBINED DISTANCE: {total_distance/1000:.2f} km")
    print(f"⏱️  TOTAL COMBINED VEHICLE TIME: {total_time/3600:.2f} hours")
    print("=" * 60)

    print("\nGenerating interactive map...")
    route_map = visualize_routes(G, routes, locations, node_coords)
    return route_map


# ============================================================================
# 6.  EXECUTE
# ============================================================================

map_object = main()
map_object

All libraries imported successfully!
  Nodes: 5229, Edges: 12292
Selected 1 depot and 30 delivery points.
Computing distance matrix...
  ⚠️ No road path between 0 and 23, using Euclidean: 3290m
  ⚠️ No road path between 1 and 23, using Euclidean: 4477m
  ⚠️ No road path between 2 and 23, using Euclidean: 4633m
  ⚠️ No road path between 3 and 23, using Euclidean: 3907m
  ⚠️ No road path between 4 and 23, using Euclidean: 3186m
  ⚠️ No road path between 5 and 23, using Euclidean: 5566m
  ⚠️ No road path between 6 and 23, using Euclidean: 4134m
  ⚠️ No road path between 7 and 23, using Euclidean: 6547m
  ⚠️ No road path between 8 and 23, using Euclidean: 7402m
  ⚠️ No road path between 9 and 23, using Euclidean: 7468m
  ⚠️ No road path between 10 and 23, using Euclidean: 7736m
  ⚠️ No road path between 11 and 23, using Euclidean: 4982m
  ⚠️ No road path between 12 and 23, using Euclidean: 6838m
  ⚠️ No road path between 13 and 23, using Euclidean: 4937m
  ⚠️ No road path between 14 and 23

In [ ]:
# -*- coding: utf-8 -*-
"""Time‑Aware Capacitated Vehicle Routing Problem – Structured (Fixed)

Now uses Euclidean‑based time fallback when no road path exists, ensuring feasibility.
"""

# ============================================================================
# 1.  INSTALLATION (uncomment and run once in Colab)
# ============================================================================
# !pip install osmnx networkx folium ortools

# ============================================================================
# 2.  IMPORTS
# ============================================================================
import osmnx as ox
import networkx as nx
import folium
import numpy as np
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

print("All libraries imported successfully!")

# ============================================================================
# 3.  CONFIGURATION
# ============================================================================
CENTER_POINT = (28.6129, 77.2295)   # India Gate, Delhi
RADIUS_METERS = 4000
NETWORK_TYPE = 'drive'

NUM_DELIVERIES = 30
RANDOM_SEED = 42

MAX_DEMAND = 9
SERVICE_TIME_PER_UNIT = 5 * 60      # seconds

VEHICLE_CAPACITIES = [85, 60, 65]
MAX_ROUTE_TIME = 14 * 60 * 60       # 14 hours

SOLVER_TIME_LIMIT_SECONDS = 10
FALLBACK_SPEED_KPH = 30             # km/h for Euclidean fallback

# ============================================================================
# 4.  HELPER FUNCTIONS
# ============================================================================

def load_road_network(center, radius, network_type):
    print(f"Downloading road network (radius {radius/1000:.1f} km) ...")
    G = ox.graph_from_point(center, dist=radius, network_type=network_type)
    print(f"  Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
    return G


def select_locations(G, num_deliveries, seed=None):
    if seed is not None:
        np.random.seed(seed)
    all_nodes = list(G.nodes)
    depot = np.random.choice(all_nodes, 1)[0]
    candidates = [n for n in all_nodes if n != depot]
    deliveries = np.random.choice(candidates, num_deliveries, replace=False)
    locations = [depot] + list(deliveries)
    node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in locations}
    print(f"Selected 1 depot and {num_deliveries} delivery points.")
    return locations, node_coords


def compute_distance_matrix(G, locations):
    n = len(locations)
    dist_matrix = np.zeros((n, n), dtype=np.int64)
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                dist_matrix[i][j] = 0
                continue
            try:
                path_length = nx.shortest_path_length(
                    G, locations[i], locations[j], weight='length'
                )
                dist_matrix[i][j] = int(path_length)
            except nx.NetworkXNoPath:
                fallback = euclidean_distance(i, j)
                dist_matrix[i][j] = fallback
                print(f"  ⚠️ No road path between {i} and {j}, using Euclidean: {fallback}m")
    return dist_matrix


def compute_travel_time_matrix(G, locations, fallback_speed_kph=30):
    """
    Compute travel times in seconds. If no road path exists, use Euclidean
    distance / (fallback speed) as a fallback.
    """
    # Add travel_time attribute to edges
    G_time = G.copy()
    for u, v, k, data in G_time.edges(keys=True, data=True):
        distance_m = data.get('length', 0)
        speed_kph = data.get('speed_kph', fallback_speed_kph)
        if isinstance(speed_kph, list):
            speed_kph = speed_kph[0]
        try:
            speed_kph = float(speed_kph)
        except (TypeError, ValueError):
            speed_kph = fallback_speed_kph
        speed_mps = speed_kph * 1000 / 3600
        travel_time = distance_m / speed_mps if speed_mps > 0 else 0
        data['travel_time'] = int(travel_time)

    n = len(locations)
    time_matrix = np.zeros((n, n), dtype=np.int64)

    # Precompute coords for Euclidean fallback
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                time_matrix[i][j] = 0
                continue
            try:
                tt = nx.shortest_path_length(
                    G_time, locations[i], locations[j], weight='travel_time'
                )
                time_matrix[i][j] = int(tt)
            except nx.NetworkXNoPath:
                # Fallback: Euclidean distance / fallback speed (m/s)
                distance_m = euclidean_distance(i, j)
                speed_mps = fallback_speed_kph * 1000 / 3600
                fallback_time = int(distance_m / speed_mps)
                time_matrix[i][j] = fallback_time
                print(f"  ⚠️ No travel‑time path between {i} and {j}, using Euclidean time: {fallback_time}s")
    return time_matrix


def generate_demands(num_locations, seed=None, max_demand=9):
    if seed is not None:
        np.random.seed(seed)
    demands = [0] + list(np.random.randint(1, max_demand+1, size=num_locations-1))
    # Convert to plain Python ints for cleaner printing
    demands = [int(d) for d in demands]
    print(f"Generated demands (total = {sum(demands)}): {demands}")
    return demands


def generate_service_times(demands, time_per_unit):
    service = [0] + [d * time_per_unit for d in demands[1:]]
    service = [int(s) for s in service]
    print(f"Service times (seconds): {service}")
    return service


def solve_time_aware_cvrp(time_matrix, demands, service_times,
                          vehicle_capacities, max_route_time,
                          time_limit=10, depot=0):
    num_vehicles = len(vehicle_capacities)
    num_nodes = len(time_matrix)

    data = {
        'time_matrix': time_matrix,
        'demands': demands,
        'service_times': service_times,
        'vehicle_capacities': vehicle_capacities,
        'num_vehicles': num_vehicles,
        'depot': depot,
        'max_route_time': max_route_time
    }

    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        travel = data['time_matrix'][from_node][to_node]
        service = data['service_times'][to_node]
        return int(travel + service)

    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity'
    )

    routing.AddDimension(
        transit_callback_index,
        0,
        data['max_route_time'],
        True,
        'Time'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = time_limit

    solution = routing.SolveWithParameters(search_parameters)
    if not solution:
        print("❌ No feasible solution found!")
        return None

    routes = {}
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(manager.IndexToNode(index))
        routes[vehicle_id] = route_nodes

    return {
        'routes': routes,
        'solution': solution,
        'routing': routing,
        'manager': manager,
        'time_dimension': routing.GetDimensionOrDie('Time')
    }


def visualize_routes(G, routes, locations, node_coords):
    depot_osm_id = locations[0]
    depot_lat, depot_lon = node_coords[depot_osm_id]

    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14,
                   tiles='CartoDB positron')

    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

    for idx, osm_id in enumerate(locations):
        lat, lon = node_coords[osm_id]
        if idx == 0:
            folium.Marker([lat, lon], popup="🏭 Depot",
                          icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:
            folium.Marker([lat, lon], popup=f"📍 Delivery {idx}",
                          icon=folium.Icon(color='gray', icon='info-sign')).add_to(m)

    for vehicle_id, route_indices in routes.items():
        color = colors[vehicle_id % len(colors)]
        route_coords = []
        for idx in route_indices:
            osm_id = locations[idx]
            lat, lon = node_coords[osm_id]
            route_coords.append((lat, lon))
        folium.PolyLine(route_coords, color=color, weight=4, opacity=0.7,
                        popup=f"🚚 Truck {vehicle_id+1}").add_to(m)

    return m


# ============================================================================
# 5.  MAIN
# ============================================================================

def main():
    G = load_road_network(CENTER_POINT, RADIUS_METERS, NETWORK_TYPE)

    locations, node_coords = select_locations(G, NUM_DELIVERIES, RANDOM_SEED)

    print("Computing distance matrix...")
    distance_matrix = compute_distance_matrix(G, locations)
    print(f"Distance matrix shape: {distance_matrix.shape}")

    print("Computing travel time matrix (with Euclidean fallback)...")
    time_matrix = compute_travel_time_matrix(G, locations, FALLBACK_SPEED_KPH)
    print(f"Time matrix shape: {time_matrix.shape}")
    print("First 5x5 entries (seconds):")
    print(time_matrix[:5, :5])

    demands = generate_demands(len(locations), RANDOM_SEED, MAX_DEMAND)
    service_times = generate_service_times(demands, SERVICE_TIME_PER_UNIT)

    print(f"\nSolving CVRP with {len(VEHICLE_CAPACITIES)} trucks, "
          f"capacities {VEHICLE_CAPACITIES}, max route time {MAX_ROUTE_TIME/3600:.1f} h ...")
    result = solve_time_aware_cvrp(
        time_matrix=time_matrix,
        demands=demands,
        service_times=service_times,
        vehicle_capacities=VEHICLE_CAPACITIES,
        max_route_time=MAX_ROUTE_TIME,
        time_limit=SOLVER_TIME_LIMIT_SECONDS
    )

    if result is None:
        print("Exiting due to no solution.")
        return

    routes = result['routes']
    solution = result['solution']
    routing = result['routing']
    manager = result['manager']
    time_dimension = result['time_dimension']

    total_distance = 0
    total_time = 0
    print("\n" + "=" * 60)
    for v_id, route in routes.items():
        load = sum(demands[node] for node in route)
        dist = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        total_distance += dist
        end_index = routing.End(v_id)
        route_time = solution.Value(time_dimension.CumulVar(end_index))
        total_time += route_time

        print(f"Truck {v_id+1}: {route}")
        print(f"  Load: {load} / {VEHICLE_CAPACITIES[v_id]}, "
              f"Distance: {dist/1000:.2f} km, "
              f"Time: {route_time/3600:.2f} h")
    print("=" * 60)
    print(f"📏 TOTAL COMBINED DISTANCE: {total_distance/1000:.2f} km")
    print(f"⏱️  TOTAL COMBINED VEHICLE TIME: {total_time/3600:.2f} hours")
    print("=" * 60)

    print("\nGenerating interactive map...")
    route_map = visualize_routes(G, routes, locations, node_coords)
    return route_map


# ============================================================================
# 6.  EXECUTE
# ============================================================================

map_object = main()
map_object

All libraries imported successfully!
  Nodes: 5229, Edges: 12294
Selected 1 depot and 30 delivery points.
Computing distance matrix...
Distance matrix shape: (31, 31)
Computing travel time matrix (with Euclidean fallback)...
Time matrix shape: (31, 31)
First 5x5 entries (seconds):
[[  0 623 739 203 585]
 [647   0 209 574 189]
 [733  92   0 660 235]
 [192 541 660   0 506]
 [599 191 235 531   0]]
Generated demands (total = 159): [0, 7, 4, 8, 5, 7, 3, 7, 8, 5, 4, 8, 8, 3, 6, 5, 2, 8, 6, 2, 5, 1, 6, 9, 1, 3, 7, 4, 9, 3, 5]
Service times (seconds): [0, 2100, 1200, 2400, 1500, 2100, 900, 2100, 2400, 1500, 1200, 2400, 2400, 900, 1800, 1500, 600, 2400, 1800, 600, 1500, 300, 1800, 2700, 300, 900, 2100, 1200, 2700, 900, 1500]

Solving CVRP with 3 trucks, capacities [85, 60, 65], max route time 14.0 h ...

Truck 1: [0, 30, 22, 20, 11, 9, 27, 8, 10, 12, 7, 24, 2, 1, 6, 26, 0]
  Load: 82 / 85, Distance: 28.40 km, Time: 7.73 h
Truck 2: [0, 15, 4, 23, 17, 19, 29, 16, 0]
  Load: 34 / 60, Distance: 20.

In [ ]:
# -*- coding: utf-8 -*-
"""Time‑Aware Capacitated Vehicle Routing Problem – Structured (Fixed)

Now uses Euclidean‑based time fallback when no road path exists, ensuring feasibility.
"""

# ============================================================================
# 1.  INSTALLATION (uncomment and run once in Colab)
# ============================================================================
# !pip install osmnx networkx folium ortools

# ============================================================================
# 2.  IMPORTS
# ============================================================================
import osmnx as ox
import networkx as nx
import folium
import numpy as np
import requests
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

print("All libraries imported successfully!")

# ============================================================================
# 3.  CONFIGURATION
# ============================================================================
CENTER_POINT = (28.6129, 77.2295)   # India Gate, Delhi
RADIUS_METERS = 4000
NETWORK_TYPE = 'drive'

NUM_DELIVERIES = 30
RANDOM_SEED = 42

MAX_DEMAND = 9
SERVICE_TIME_PER_UNIT = 5 * 60      # seconds

VEHICLE_CAPACITIES = [85, 60, 65]
MAX_ROUTE_TIME = 14 * 60 * 60       # 14 hours

SOLVER_TIME_LIMIT_SECONDS = 10
FALLBACK_SPEED_KPH = 30             # km/h for Euclidean fallback

# ============================================================================
# 4.  HELPER FUNCTIONS
# ============================================================================

def load_road_network(center, radius, network_type):
    print(f"Downloading road network (radius {radius/1000:.1f} km) ...")
    G = ox.graph_from_point(center, dist=radius, network_type=network_type)
    print(f"  Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
    return G


def select_locations(G, num_deliveries, seed=None):
    if seed is not None:
        np.random.seed(seed)
    all_nodes = list(G.nodes)
    depot = np.random.choice(all_nodes, 1)[0]
    candidates = [n for n in all_nodes if n != depot]
    deliveries = np.random.choice(candidates, num_deliveries, replace=False)
    locations = [depot] + list(deliveries)
    node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in locations}
    print(f"Selected 1 depot and {num_deliveries} delivery points.")
    return locations, node_coords


def compute_distance_matrix(G, locations):
    n = len(locations)
    dist_matrix = np.zeros((n, n), dtype=np.int64)
    coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in locations]

    def euclidean_distance(i, j):
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                dist_matrix[i][j] = 0
                continue
            try:
                path_length = nx.shortest_path_length(
                    G, locations[i], locations[j], weight='length'
                )
                dist_matrix[i][j] = int(path_length)
            except nx.NetworkXNoPath:
                fallback = euclidean_distance(i, j)
                dist_matrix[i][j] = fallback
                print(f"  ⚠️ No road path between {i} and {j}, using Euclidean: {fallback}m")
    return dist_matrix



def compute_travel_time_matrix(G, locations, node_coords, fallback_speed_kph=30):
    """
    Try OSRM public API first (free, accurate). Fallback to OSMnx if unavailable.
    """
    try:
        # Build OSRM coordinate string
        coord_pairs = []
        for node in locations:
            lat, lon = node_coords[node]   # node_coords is passed from main
            coord_pairs.append(f"{lon},{lat}")
        coord_str = ";".join(coord_pairs)

        url = f"http://router.project-osrm.org/table/v1/driving/{coord_str}?annotations=duration"
        response = requests.get(url, timeout=5)
        data = response.json()

        if 'durations' in data:
            time_matrix = np.array(data['durations'], dtype=np.int64)
            print("✅ Travel time matrix fetched from OSRM API (free!)")
            return time_matrix
        else:
            print("⚠️ OSRM API error, falling back to OSMnx.")
    except Exception as e:
        print(f"⚠️ OSRM request failed ({e}), falling back to OSMnx.")

    # ---------- FALLBACK: Your existing OSMnx method (improved) ----------
    # Add realistic speeds if not already present
    if 'speed_kph' not in G.edges(data=True):
        custom_speeds = {
            'motorway': 50, 'trunk': 40, 'primary': 30,
            'secondary': 25, 'tertiary': 20, 'residential': 15,
            'living_street': 10, 'service': 15, 'unclassified': 20
        }
        G = ox.add_edge_speeds(G, hwy_speeds=custom_speeds)

    G_time = G.copy()
    for u, v, k, data in G_time.edges(keys=True, data=True):
        distance_m = data.get('length', 0)
        speed_kph = data.get('speed_kph', fallback_speed_kph)
        if isinstance(speed_kph, list):
            speed_kph = speed_kph[0]
        try:
            speed_kph = float(speed_kph)
        except (TypeError, ValueError):
            speed_kph = fallback_speed_kph
        speed_mps = speed_kph * 1000 / 3600
        travel_time = distance_m / speed_mps if speed_mps > 0 else 0
        # Add congestion factor and turn penalty for realism
        travel_time = travel_time * 1.3 + 5   # 30% congestion + 5 sec per edge
        data['travel_time'] = int(travel_time)

    n = len(locations)
    time_matrix = np.zeros((n, n), dtype=np.int64)
    for i in range(n):
        for j in range(n):
            if i == j:
                time_matrix[i][j] = 0
                continue
            try:
                tt = nx.shortest_path_length(
                    G_time, locations[i], locations[j], weight='travel_time'
                )
                time_matrix[i][j] = int(tt)
            except nx.NetworkXNoPath:
                # Use Euclidean fallback (as before)
                lat1, lon1 = node_coords[locations[i]]
                lat2, lon2 = node_coords[locations[j]]
                dist = int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)
                speed_mps = fallback_speed_kph * 1000 / 3600
                time_matrix[i][j] = int(dist / speed_mps)
    return time_matrix


def generate_demands(num_locations, seed=None, max_demand=9):
    if seed is not None:
        np.random.seed(seed)
    demands = [0] + list(np.random.randint(1, max_demand+1, size=num_locations-1))
    # Convert to plain Python ints for cleaner printing
    demands = [int(d) for d in demands]
    print(f"Generated demands (total = {sum(demands)}): {demands}")
    return demands


def generate_service_times(demands, time_per_unit):
    service = [0] + [d * time_per_unit for d in demands[1:]]
    service = [int(s) for s in service]
    print(f"Service times (seconds): {service}")
    return service


def solve_time_aware_cvrp(time_matrix, demands, service_times,
                          vehicle_capacities, max_route_time,
                          time_limit=10, depot=0):
    num_vehicles = len(vehicle_capacities)
    num_nodes = len(time_matrix)

    data = {
        'time_matrix': time_matrix,
        'demands': demands,
        'service_times': service_times,
        'vehicle_capacities': vehicle_capacities,
        'num_vehicles': num_vehicles,
        'depot': depot,
        'max_route_time': max_route_time
    }

    manager = pywrapcp.RoutingIndexManager(num_nodes, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        travel = data['time_matrix'][from_node][to_node]
        service = data['service_times'][to_node]
        return int(travel + service)

    transit_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,
        data['vehicle_capacities'],
        True,
        'Capacity'
    )

    routing.AddDimension(
        transit_callback_index,
        0,
        data['max_route_time'],
        True,
        'Time'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = time_limit

    solution = routing.SolveWithParameters(search_parameters)
    if not solution:
        print("❌ No feasible solution found!")
        return None

    routes = {}
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(manager.IndexToNode(index))
        routes[vehicle_id] = route_nodes

    return {
        'routes': routes,
        'solution': solution,
        'routing': routing,
        'manager': manager,
        'time_dimension': routing.GetDimensionOrDie('Time')
    }


def visualize_routes(G, routes, locations, node_coords):
    depot_osm_id = locations[0]
    depot_lat, depot_lon = node_coords[depot_osm_id]

    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14,
                   tiles='CartoDB positron')

    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

    for idx, osm_id in enumerate(locations):
        lat, lon = node_coords[osm_id]
        if idx == 0:
            folium.Marker([lat, lon], popup="🏭 Depot",
                          icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:
            folium.Marker([lat, lon], popup=f"📍 Delivery {idx}",
                          icon=folium.Icon(color='gray', icon='info-sign')).add_to(m)

    for vehicle_id, route_indices in routes.items():
        color = colors[vehicle_id % len(colors)]
        route_coords = []
        for idx in route_indices:
            osm_id = locations[idx]
            lat, lon = node_coords[osm_id]
            route_coords.append((lat, lon))
        folium.PolyLine(route_coords, color=color, weight=4, opacity=0.7,
                        popup=f"🚚 Truck {vehicle_id+1}").add_to(m)

    return m


# ============================================================================
# 5.  MAIN
# ============================================================================

def main():
    G = load_road_network(CENTER_POINT, RADIUS_METERS, NETWORK_TYPE)

    locations, node_coords = select_locations(G, NUM_DELIVERIES, RANDOM_SEED)

    print("Computing distance matrix...")
    distance_matrix = compute_distance_matrix(G, locations)
    print(f"Distance matrix shape: {distance_matrix.shape}")

    print("Computing travel time matrix (with Euclidean fallback)...")
    time_matrix = compute_travel_time_matrix(G, locations, node_coords, FALLBACK_SPEED_KPH)
    print(f"Time matrix shape: {time_matrix.shape}")
    print("First 5x5 entries (seconds):")
    print(time_matrix[:5, :5])

    demands = generate_demands(len(locations), RANDOM_SEED, MAX_DEMAND)
    service_times = generate_service_times(demands, SERVICE_TIME_PER_UNIT)

    print(f"\nSolving CVRP with {len(VEHICLE_CAPACITIES)} trucks, "
          f"capacities {VEHICLE_CAPACITIES}, max route time {MAX_ROUTE_TIME/3600:.1f} h ...")
    result = solve_time_aware_cvrp(
        time_matrix=time_matrix,
        demands=demands,
        service_times=service_times,
        vehicle_capacities=VEHICLE_CAPACITIES,
        max_route_time=MAX_ROUTE_TIME,
        time_limit=SOLVER_TIME_LIMIT_SECONDS
    )

    if result is None:
        print("Exiting due to no solution.")
        return

    routes = result['routes']
    solution = result['solution']
    routing = result['routing']
    manager = result['manager']
    time_dimension = result['time_dimension']

    total_distance = 0
    total_time = 0
    print("\n" + "=" * 60)
    for v_id, route in routes.items():
        load = sum(demands[node] for node in route)
        dist = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        total_distance += dist
        end_index = routing.End(v_id)
        route_time = solution.Value(time_dimension.CumulVar(end_index))
        total_time += route_time

        print(f"Truck {v_id+1}: {route}")
        print(f"  Load: {load} / {VEHICLE_CAPACITIES[v_id]}, "
              f"Distance: {dist/1000:.2f} km, "
              f"Time: {route_time/3600:.2f} h")
    print("=" * 60)
    print(f"📏 TOTAL COMBINED DISTANCE: {total_distance/1000:.2f} km")
    print(f"⏱️  TOTAL COMBINED VEHICLE TIME: {total_time/3600:.2f} hours")
    print("=" * 60)

    print("\nGenerating interactive map...")
    route_map = visualize_routes(G, routes, locations, node_coords)
    return route_map


# ============================================================================
# 6.  EXECUTE
# ============================================================================

map_object = main()
map_object

All libraries imported successfully!
  Nodes: 5229, Edges: 12294
Selected 1 depot and 30 delivery points.
Computing distance matrix...
Distance matrix shape: (31, 31)
Computing travel time matrix (with Euclidean fallback)...
✅ Travel time matrix fetched from OSRM API (free!)
Time matrix shape: (31, 31)
First 5x5 entries (seconds):
[[  0 507 555 201 434]
 [453   0 162 426 128]
 [500  64   0 473 174]
 [189 446 494   0 373]
 [433 130 178 406   0]]
Generated demands (total = 159): [0, 7, 4, 8, 5, 7, 3, 7, 8, 5, 4, 8, 8, 3, 6, 5, 2, 8, 6, 2, 5, 1, 6, 9, 1, 3, 7, 4, 9, 3, 5]
Service times (seconds): [0, 2100, 1200, 2400, 1500, 2100, 900, 2100, 2400, 1500, 1200, 2400, 2400, 900, 1800, 1500, 600, 2400, 1800, 600, 1500, 300, 1800, 2700, 300, 900, 2100, 1200, 2700, 900, 1500]

Solving CVRP with 3 trucks, capacities [85, 60, 65], max route time 14.0 h ...

Truck 1: [0, 30, 22, 27, 9, 8, 10, 7, 12, 11, 20, 6, 26, 0]
  Load: 70 / 85, Distance: 25.72 km, Time: 6.54 h
Truck 2: [0, 25, 28, 18, 14, 5, 